# Zwicky Transient Facility (ZTF)

El propósito de este notebook es analizar la muestra total del último Data Release de ZTF, su tamaño y los objetos que contiene. Luego, limpiar y depurar los datos basándose en los criterios del artículo **Unlocking AGN Variability with Custom ZTF Photometry for
High-Fidelity Light Curves and Robust Selection**. 
Una vez seleccionados los datos útiles para el trabajo, se realizará un histograma de longitud temporal por objeto longitud temporal (baseline) . Luego, un histograma de la magnitud media por objeto, calculada por separado para filtro g y filtro r promediadas sobre todas las épocas de ese objeto, graficadas como dos curvas superpuestas sobre el mismo eje x (magnitud media).

La meta final es elaborar un algoritmo que resuelva y estudie la precesión de AGNs, para ser aplicado en un futuro a datos del LSST. 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 

# Librerías de astronomía y acceso a catálogos
from astropy.table import Table
from astropy.coordinates import SkyCoord
from astropy import units as u

# Para consultar IRSA (donde vive ZTF) por TAP
import pyvo as vo
# alternativa más "amigable" de astroquery:
from astroquery.ipac.irsa import Irsa




In [ ]:
TAP = vo.dal.TAPService("https://irsa.ipac.caltech.edu/TAP")

# Veamos qué tablas tiene ZTF
QUERY_tables = """
SELECT table_name, description 
FROM TAP_SCHEMA.tables
WHERE table_name LIKE '%ztf%'
"""
tables = TAP.search(QUERY_tables).to_table()
print(tables)

        table_name                  description           
------------------------- --------------------------------
 ztf.ztf_current_path_raw            ZTF Raw Product Paths
 ztf.ztf_current_path_cal    ZTF Calibration Product Paths
ztf.ztf_current_path_deep ZTF Deep Reference Product Paths
         ztf_objects_dr24                 ZTF DR24 Objects
         ztf_objects_dr23                 ZTF DR23 Objects
         ztf_objects_dr22                 ZTF DR22 Objects
         ztf_objects_dr21                 ZTF DR21 Objects
         ztf_objects_dr20                 ZTF DR20 Objects
 ztf.ztf_current_meta_sci      ZTF Science Exposure Images
 ztf.ztf_current_meta_ref     ZTF Reference (coadd) Images
 ztf.ztf_current_meta_raw           ZTF Raw Metadata Table
 ztf.ztf_current_meta_cal   ZTF Calibration Metadata Table
ztf.ztf_current_meta_deep        ZTF Deep Reference Images
 ztf.ztf_current_path_sci        ZTF Science Product Paths
 ztf.ztf_current_path_ref      ZTF Reference Product Pat

La tabla que queremos es ztf_objects_dr24 porque es el Data Release más reciente, y por tanto el más completo y actualizado. 

In [ ]:
query_columns = """
SELECT column_name, datatype, description, unit
FROM TAP_SCHEMA.columns
WHERE table_name = 'ztf_objects_dr24'
ORDER BY column_index
"""
cols = TAP.search(query_columns).to_table().to_pandas()
pd.set_option('display.max_rows', None)
cols

                column_name datatype  \
0                      cntr     long   
1                       oid     long   
2                        ra   double   
3                       dec   double   
4                     htm20     long   
5                     field      int   
6                     ccdid    short   
7                       qid    short   
8                       fid    short   
9                filtercode     char   
10                        x   double   
11                        y   double   
12                        z   double   
13                 ngoodobs      int   
14              ngoodobsrel      int   
15                     nobs      int   
16                  nobsrel      int   
17                   refchi    float   
18                   refmag    float   
19                refmagerr    float   
20                 refsharp    float   
21                   refsnr    float   
22           astrometricrms   double   
23                    chisq    float   


En la sección 2, pág 2 del paper de referencia dicen "we limited the download to −29° < dec < +15°, where −29° is the southern limit of ZTF, and to Galactic latitudes |b| ≳ 20° above and below the Galactic plane."
Aplicaré los mismos criterios, que son básicamente:
- Declinación: −29° < dec < +15°
- Latitud galáctica: |b| ≳ 20° (por encima y por debajo del plano galáctico)

In [9]:
columns = """
oid, ra, dec, fid, filtercode,
nobs, ngoodobs,
meanmag, medianmag, magrms, weightedmeanmag, weightedmagrms,
chisq, con, lineartrend,
skewness, smallkurtosis,
stetsonj, stetsonk, vonneumannratio,
refmag, refmagerr, refsnr
"""

In [15]:
query = f"""
SELECT TOP 100000 {columns}
FROM ztf_objects_dr24
WHERE dec > -29 AND dec < 15
"""
job = TAP.submit_job(query)
job.run()
job.wait()
result = job.fetch_result().to_table().to_pandas()

In [16]:
result.head()

,oid,ra,dec,fid,filtercode,nobs,ngoodobs,meanmag,medianmag,magrms,...,con,lineartrend,skewness,smallkurtosis,stetsonj,stetsonk,vonneumannratio,refmag,refmagerr,refsnr
0,1377210400175199,283.477552,-13.119184,2,zr,24,19,20.087191,20.089390,0.141067,...,1.0,0.000024,-0.254074,NaN,NaN,NaN,1.783238,19.966999,0.072,15.00
1,384201400157518,283.477569,-13.119182,2,zr,437,221,20.070490,20.083590,0.131142,...,1.0,-0.000020,-0.101622,NaN,NaN,NaN,1.814590,19.919001,0.072,15.14
2,384301300150053,283.477598,-13.119165,3,zi,96,15,19.543711,19.526461,0.109098,...,1.0,0.000492,-0.061225,NaN,NaN,NaN,2.541256,19.523001,0.108,10.06
3,1377210400194730,283.480131,-13.117467,2,zr,32,27,19.120899,19.157669,0.141483,...,1.0,-0.000040,-0.887339,NaN,NaN,NaN,1.106052,19.023001,0.102,10.64
4,384101400156313,283.480176,-13.117414,1,zg,128,111,19.826281,19.829370,0.163416,...,1.0,-0.000007,0.056804,NaN,NaN,NaN,2.338639,19.867001,0.068,15.99
